In [1]:
import gurobipy as gp
from gurobipy import GRB, quicksum
import pandas as pd
import numpy as np
import csv

In [2]:
NAMES = {
    0: "Start / Banalacan Port",
    1: "Adeling's Karenderia",
    2: "K N's Kitchen",
    3: "Street 74 Restaurant",
    4: "Geodetic Center",
    5: "Villa Negro Café",
    6: "Paadjao Falls",
    7: "Tarug Cave",
    8: "Bagumbungan",
    9: "Bathala Cave",
    10: "Kawa-Kawa Falls",
    11: "Holy Cross Parish Church",
    12: "Rico's Inn",
    13: "Vdine Restaurant",
    14: "Stop'n Shop",
    15: "Oromismo Hotel 898",
    16: "Aler Café",
    17: "Carmen Eco-Park",
    18: "Pulang Lupa",
    19: "Cate Tanawin",
    20: "Ka Amon Cave",
    21: "Poctoy White Beach Resort",
    22: "Mang Bens",
    23: "Ajumma",
    24: "Amirah's Place",
    25: "Em's Food House 2",
    26: "Meal Tea Wings",
    27: "Rosales One Stop Shop",
    28: "Malbog Hotspring",
    29: "Kape Dring",
    30: "Mss Hotel And Resto",
    31: "Green Crib",
    32: "Tambayan Sa Curba",
    33: "Curba Grill",
    34: "St. Joseph The Worker Parish Church",
    35: "Bayview",
    36: "Sinugba",
    37: "Unica Ija",
    38: "Marigold",
    39: "Monsan Brew",
    40: "Talao Cave",
    41: "Guingona Park",
    42: "Tita Lida's",
    43: "LUXOR Hotel and Restaurant",
    44: "Marinduque Airport",
    45: "Cumba",
    46: "Theresa Store/Karinderia",
    47: "Duyay Cave",
    48: "Boac Cathedral",
    49: "The Penthouse",
    50: "Eomma",
    51: "Bricks And Coals",
    52: "10 Y.O.",
    53: "Cravings",
    54: "Tony And Aida's Pansiteria",
    55: "Lola Pia's Pansit",
    56: "Cafe Benevolo",
    57: "Cafe Tolyo",
    58: "Porknok Putok Batok",
    59: "Khendal Foodhub",
    60: "Hale Al",
    61: "4900",
    62: "Uncle Bhogs",
    63: "Chinggay's Café",
    64: "Curba Grill",
    65: "Blue Building Food Court",
    66: "Kusina Sa Plaza",
    67: "Good Chow Food Express",
    68: "La Concha",
    69: "Picaro",
    70: "Lucky Food House",
    71: "Monserrat",
    72: "Meal Tea Wings Murallon",
    73: "Marinduque Museum",
    74: "Marinduque Wildlife Sanctuary",
    75: "St.Isidore Parish Church",
    76: "Las Hermana's Restaurant",
    77: "Tonicita's",
    78: "Poblacion Pares",
    79: "Calle Labao Diner",
    80: "Bethany Cafe"
}

RES = [1, 2, 3, 5, 12, 13, 14, 15, 16, 19, 22, 23, 24, 25, 26, 27, 29,
    30, 31, 32, 33, 35, 36, 37, 38, 39, 42, 43, 45, 46, 49, 50, 51, 52,
    53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69,
    70, 71, 72, 76, 77, 78, 79, 80]

In [3]:
df = pd.read_csv("./times.csv",encoding="latin1").values
s = pd.read_csv("./stay.csv",encoding="latin1").values[:,1]
sched = pd.read_csv("./scheds.csv",encoding="latin1").values

In [4]:
# ---------- Parameters ----------
START = 8                #start time in military time
TIME_LIMIT = 480           #total tour time in minutes
START_EAT = 11             #restaurnt break time window start
END_EAT = 14               #restaurnt break time window end 
START_NODE = 0             #either 0 or 44
theme = "Church"           # Historical, Water, Church, Nature, Cave or None
Custom_Priority = set([])    # Custom priority nodes, can be empty

In [6]:
# ---------- Sets ----------
n = df.shape[0]                          # number of nodes
N = [j for j in range(1, n) if j != 44]  #generate list of possible locations, 0 is BATACLAN PORT, 44 is the Marinduque airport
N0 = [START_NODE] + N                    # origin + all others

c = {(i, j): float(df[i][j]) for i in range(n) for j in range(n)}  #time-distance matrix

for i in range(len(sched)):              #visiting time windows
    if i in RES:
        sched[i,1:3] = [START_EAT,END_EAT]
OPEN = sched[:,1]
CLOSE = sched[:,2]


Historical = {4,18,73}                   #references to themed tours
Water   = {6,10,21,28}
Church     = {11,34,48,75}
Nature     = {17,41,74}
Cave       = {7,8,9,40,20,47} 

if theme == "Historical":
    P = Historical
elif theme == "Cave":
    P = Cave
elif theme == "Water":
    P = Water
elif theme == "Church":
    P = Church
elif theme == "Nature":
    P = Nature
else:
    P = Custom_Priority

RESTAURANT_NODES = {i: 1 if i in RES else 0 for i in N}   #references to restaurant nodes

m = gp.Model("TourPlanning")                              #build the model
x = m.addVars(N0, N0, vtype=GRB.BINARY, name="x")
u = m.addVars(N, vtype=GRB.INTEGER, lb=1, ub=len(N)-1, name="u")
t = m.addVars(N0, vtype=GRB.CONTINUOUS, lb=0, ub=TIME_LIMIT, name="t")

m.setObjective(quicksum(x[i, j] for i in N for j in N if i != j), GRB.MAXIMIZE)     #objective function

m.addConstr(quicksum(x[START_NODE, j] for j in N) == 1, "Start_From_Origin")        #constraint: start at origin

m.addConstrs(                                                                       #constraint: flow conservation  
    (quicksum(x[i, j] for i in N0 if i != j) ==
     quicksum(x[j, k] for k in N0 if k != j)) for j in N
)

m.addConstrs(                                                                       #constraint: visit at most once
    (quicksum(x[i, j] for i in N0 if i != j) <= 1) for j in N
)

m.addConstr(                                                                        #constraint: time limit
    quicksum(c[i, j] * x[i, j] for i in N0 for j in N0 if i != j)
  + quicksum(s[i] * quicksum(x[i, j] for j in N0 if j != i) for i in N0)
  <= TIME_LIMIT,
    "Total_Time_Limit"
)

m.addConstrs(                                                                       #constraint: subtour elimination 
    (u[i] - u[j] + (len(N)-1) * x[i, j] <= len(N)-2)
    for i in N for j in N if i != j
)

m.addConstrs(                                                                       #constraint: visit theme nodes
    (quicksum(x[i, p] for i in N0 if i != p) == 1) for p in P
)


m.addConstr(                                                                        #constraint: visit restaurant   
    quicksum(x[i, j] for i in N0 for j in N if i != j and RESTAURANT_NODES[j] == 1) == 1,
    "Only_One_Restaurant_Node"
)

#arrival time propagation
M = TIME_LIMIT  
m.addConstr(t[START_NODE] == 0, "Init_Time")
m.addConstrs(
    (t[j] >= t[i] + c[i, j] + s[i] - M*(1 - x[i, j]))
    for i in N0 for j in N if i != j
)
m.addConstrs(
    (t[j] <= t[i] + c[i, j] + s[i] + M*(1 - x[i, j]))
    for i in N0 for j in N if i != j
)

#arrival time constraints for all locations
M1 = START+TIME_LIMIT/60  #BIG M
for j in N:
    m.addConstr(
        START + t[j]/60 >= OPEN[j] - M1*(1 - quicksum(x[i,j] for i in N0 if i != j)),
        name=f"open_time_{j}"
    )
    m.addConstr(
        START + (t[j] + s[j])/60 <= CLOSE[j] + M1*(1 - quicksum(x[i,j] for i in N0 if i != j)),
        name=f"close_time_{j}"
    )
            
#exactly one return to the origin
m.addConstr(quicksum(x[i, START_NODE] for i in N) == 1, "ReturnToOrigin")

#m.setParam("PoolSearchMode", 1)
#m.setParam("PoolSolutions", 100)
m.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 18969 rows, 6559 columns and 98751 nonzeros
Model fingerprint: 0xdcdf6de7
Variable types: 80 continuous, 6479 integer (6400 binary)
Coefficient statistics:
  Matrix range     [2e-02, 5e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 5e+02]
  RHS range        [1e+00, 6e+02]
Presolve removed 7064 rows and 3546 columns
Presolve time: 0.20s
Presolved: 11905 rows, 3013 columns, 43722 nonzeros
Variable types: 79 continuous, 2934 integer (2855 binary)
Found heuristic solution: objective 5.0000000
Found heuristic solution: objective 6.0000000

Root relaxation: objective 1.121574e+01, 628 iterations, 0.07 seconds (0.05 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Wo

In [8]:
if m.status == GRB.OPTIMAL:
    travel_time = sum(c[i, j] * x[i, j].x for i in N0 for j in N0 if i != j)
    service_time = sum(s[j] for j in N if any(x[i, j].x > 0.5 for i in N0 if i != j))
    total_time = travel_time + service_time

    print(f"Total Tour Duration = {total_time}")
    print("Selected edges:")
    for i in N0:
        for j in N0:
            if i != j and x[i, j].x > 0.5:
                print(f"From {i, NAMES[i]} to {j, NAMES[j]}")

Total Tour Duration = 472.33833333
Selected edges:
From (0, 'Start / Banalacan Port') to (17, 'Carmen Eco-Park')
From (11, 'Holy Cross Parish Church') to (80, 'Bethany Cafe')
From (17, 'Carmen Eco-Park') to (20, 'Ka Amon Cave')
From (18, 'Pulang Lupa') to (11, 'Holy Cross Parish Church')
From (20, 'Ka Amon Cave') to (18, 'Pulang Lupa')
From (34, 'St. Joseph The Worker Parish Church') to (41, 'Guingona Park')
From (41, 'Guingona Park') to (75, 'St.Isidore Parish Church')
From (48, 'Boac Cathedral') to (73, 'Marinduque Museum')
From (73, 'Marinduque Museum') to (34, 'St. Joseph The Worker Parish Church')
From (75, 'St.Isidore Parish Church') to (0, 'Start / Banalacan Port')
From (80, 'Bethany Cafe') to (48, 'Boac Cathedral')


In [9]:
print(f"Total Tour Duration = {total_time}")

if m.status == GRB.OPTIMAL:
    # create filename
    filename = f"{theme}_{START_NODE}.csv"

    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)

        # write total duration at the top
        writer.writerow(["Total Tour Duration", total_time])
        writer.writerow([])  # blank row for readability
        
        # write header to CSV
        writer.writerow(["Node", "Name", "Arrival", "Open", "Close"])

        # print header to console
        print(f"{'Node':<5}{'Name':<40}{'Arrival':<10}{'OPEN':<10}{'CLOSE':<10}")

        visited_nodes = []
        for j in N0:
            if any(x[i, j].X > 0.5 for i in N0 if i != j):
                visited_nodes.append((t[j].X, j))

        for i in N0:
            if x[i, START_NODE].X > 0.5:
                return_time = t[i].X + c[i, START_NODE] + s[i]
                visited_nodes.append((return_time, START_NODE))

        for arrival, j in sorted(visited_nodes):
            total_hours = START + arrival / 60
            hours = int(total_hours)
            minutes = int(round((total_hours - hours) * 60))
            arrival_str = f"{hours:02d}h{minutes:02d}m"

            open_str = f"{OPEN[j]}h"
            close_str = f"{CLOSE[j]}h"

            print(f"{j:<5}{NAMES[j]:<40}{arrival_str:<10}{open_str:<10}{close_str:<10}")

            writer.writerow([j, NAMES[j], arrival_str, open_str, close_str])

Total Tour Duration = 472.33833333
Node Name                                    Arrival   OPEN      CLOSE     
0    Start / Banalacan Port                  08h00m    0h        24h       
17   Carmen Eco-Park                         08h35m    8h        17h       
20   Ka Amon Cave                            09h18m    8h        17h       
18   Pulang Lupa                             10h37m    6h        17h       
11   Holy Cross Parish Church                11h35m    0h        24h       
80   Bethany Cafe                            12h13m    11h       14h       
48   Boac Cathedral                          13h19m    0h        24h       
73   Marinduque Museum                       13h45m    8h        17h       
34   St. Joseph The Worker Parish Church     14h22m    0h        24h       
41   Guingona Park                           14h40m    0h        24h       
75   St.Isidore Parish Church                15h31m    0h        24h       
0    Start / Banalacan Port                  15h52m  